## Load Clean Data

In [ ]:
import pandas as pd

df = pd.read_csv('../data/processed/clean_retail_data.csv.gz')

# Convert date again (safety step)
df['InvoiceDate'] = pd.to_datetime(df['InvoiceDate'])

df.head()

## Define Snapshot Date

In [ ]:
snapshot_date = df['InvoiceDate'].max() + pd.Timedelta(days=1)
snapshot_date

## Create Total Price COlumn

In [ ]:
df['TotalAmount'] = df['Quantity'] * df['Price']

## Calculate RFM Metrics

In [ ]:
rfm = df.groupby('Customer ID').agg({
    'InvoiceDate': lambda x: (snapshot_date - x.max()).days,
    'Invoice': 'nunique',
    'TotalAmount': 'sum'
})

# Rename columns
rfm.columns = ['Recency', 'Frequency', 'Monetary']

rfm.head()

## Quick Sanity Check

In [ ]:
rfm.describe()

## Save RFM Table

In [ ]:
rfm.to_csv('../data/processed/rfm_table.csv')

## Load RFM Table

In [ ]:
import pandas as pd

rfm = pd.read_csv('../data/processed/rfm_table.csv', index_col=0)

rfm.head()

## Create RFM Score (1-5)

In [ ]:
rfm['R_score'] = pd.qcut(rfm['Recency'], 5, labels=[5,4,3,2,1])

## Frequency Score

In [ ]:
rfm['F_score'] = pd.qcut(rfm['Frequency'].rank(method='first'), 5, labels=[1,2,3,4,5])

## Monetary Score

In [ ]:
rfm['M_score'] = pd.qcut(rfm['Monetary'], 5, labels=[1,2,3,4,5])

## Create RFM Score

In [ ]:
rfm['RFM_Score'] = (
    rfm['R_score'].astype(str) +
    rfm['F_score'].astype(str) +
    rfm['M_score'].astype(str)
)

rfm.head()

## Segmentation Rules

In [ ]:
def segment_customer(row):
    r = int(row['R_score'])
    f = int(row['F_score'])
    m = int(row['M_score'])
    
    if r >= 4 and f >= 4 and m >= 4:
        return 'Champions'
    elif r >= 3 and f >= 3:
        return 'Loyal Customers'
    elif r >= 4 and f <= 2:
        return 'Potential Loyalists'
    elif r <= 2 and f >= 3:
        return 'At Risk'
    elif r <= 2 and f <= 2:
        return 'Lost Customers'
    else:
        return 'Others'

## Apply Segmentation

In [ ]:
rfm['Segment'] = rfm.apply(segment_customer, axis=1)

rfm.head()

## Segmentation Distribution

In [ ]:
rfm['Segment'].value_counts()

## Segment Insights

In [ ]:
rfm.groupby('Segment').agg({
    'Recency': 'mean',
    'Frequency': 'mean',
    'Monetary': 'mean'
}).round(2)

## Save Final Dataset

In [ ]:
rfm.to_csv('../data/processed/rfm_segments.csv')